# NB_bishop_ch12_figures
## Key Figures for Bishop Chapter 12 -- Transformers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/master/Bishop_Ch_12/NB_bishop_ch12_figures.ipynb)

Generate publication-quality figures for attention, multi-head attention, positional encoding, and the transformer block.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib

matplotlib.rcParams['figure.facecolor'] = 'none'
matplotlib.rcParams['axes.facecolor'] = 'none'
matplotlib.rcParams['savefig.facecolor'] = 'none'
matplotlib.rcParams['savefig.transparent'] = True
matplotlib.rcParams['axes.grid'] = False
matplotlib.rcParams['legend.loc'] = 'upper center'
matplotlib.rcParams['legend.framealpha'] = 0.0

COLORS = {'blue': '#1A3A6E', 'red': '#CD0000', 'green': '#2E7D32', 'amber': '#B5853F'}

def save_fig(fig, name, chart_dir='../../charts'):
    for ax in fig.get_axes():
        ax.grid(False)
        legend = ax.get_legend()
        if legend:
            legend.set_bbox_to_anchor((0.5, -0.15))
            legend.set_loc('upper center')
            legend._ncols = min(len(legend.get_texts()), 4)
    fig.savefig(f'{chart_dir}/{name}.pdf', bbox_inches='tight', transparent=True)
    fig.savefig(f'{chart_dir}/{name}.png', bbox_inches='tight', transparent=True, dpi=150)
    print(f'Saved {chart_dir}/{name}.pdf and .png')

np.random.seed(42)

## Figure 12.1 -- Attention Concept

Two sentences with attention lines between words; line thickness encodes attention weight.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 3.5)
ax.axis('off')

source_words = ['The', 'cat', 'sat', 'on', 'the', 'mat']
target_words = ['Le', 'chat', 'assis', 'sur', 'le', 'tapis']

y_src, y_tgt = 3.0, 0.5
for i, w in enumerate(source_words):
    ax.text(i, y_src, w, ha='center', va='center', fontsize=13, fontweight='bold',
            color=COLORS['blue'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#D6E4F0', edgecolor=COLORS['blue'], linewidth=1.2))
for i, w in enumerate(target_words):
    ax.text(i, y_tgt, w, ha='center', va='center', fontsize=13, fontweight='bold',
            color=COLORS['red'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#F5D6D6', edgecolor=COLORS['red'], linewidth=1.2))

# Attention weights (source -> target): dominant alignments
attention_links = [
    (0, 0, 0.9), (1, 1, 0.85), (2, 2, 0.8), (3, 3, 0.75),
    (4, 4, 0.7), (5, 5, 0.9),
    (0, 4, 0.15), (1, 2, 0.1), (3, 5, 0.08), (4, 0, 0.12),
]

for s, t, w in attention_links:
    ax.plot([s, t], [y_src - 0.25, y_tgt + 0.25],
            color=COLORS['amber'], alpha=min(w + 0.15, 1.0),
            linewidth=w * 5, solid_capstyle='round')

ax.text(2.5, y_src + 0.4, 'Source (English)', ha='center', fontsize=11, color=COLORS['blue'])
ax.text(2.5, y_tgt - 0.4, 'Target (French)', ha='center', fontsize=11, color=COLORS['red'])
ax.set_title('Cross-Attention: Word Alignment', fontweight='bold', fontsize=14, pad=12)

fig.tight_layout()
save_fig(fig, 'fig12_1_attention_concept')
plt.show()

## Figure 12.7 -- Scaled Dot-Product Attention Heatmap

Matrix heatmap of attention weights between query and key tokens.

In [ ]:
tokens = ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
n = len(tokens)
d_k = 16
np.random.seed(7)
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
scores = Q @ K.T / np.sqrt(d_k)
exp_s = np.exp(scores - scores.max(axis=-1, keepdims=True))
attn_weights = exp_s / exp_s.sum(axis=-1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(attn_weights, cmap='Blues', aspect='auto', vmin=0, vmax=attn_weights.max())
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(tokens, fontsize=10)
ax.set_xlabel('Key Tokens', fontsize=12)
ax.set_ylabel('Query Tokens', fontsize=12)
ax.set_title('Scaled Dot-Product Attention Weights', fontweight='bold', fontsize=14)

for i in range(n):
    for j in range(n):
        val = attn_weights[i, j]
        color = 'white' if val > 0.2 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Attention Weight', fontsize=10)

fig.tight_layout()
save_fig(fig, 'fig12_7_scaled_attention')
plt.show()

## Figure 12.8 -- Multi-Head Attention Diagram

Multiple attention heads feeding into concatenation and linear projection.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(-1, 15)
ax.set_ylim(-0.5, 8)
ax.axis('off')

head_colors = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['amber']]
head_labels = ['Head 1', 'Head 2', 'Head 3', 'Head h']

# Input box
input_box = mpatches.FancyBboxPatch((5.5, 0), 3, 0.8, boxstyle='round,pad=0.1',
                                     facecolor='#E8E8E8', edgecolor='black', linewidth=1.5)
ax.add_patch(input_box)
ax.text(7, 0.4, 'Input X', ha='center', va='center', fontsize=12, fontweight='bold')

# Linear projections Q, K, V per head
head_x_positions = [1.5, 4.5, 8.5, 11.5]
y_proj = 2.0
y_attn = 4.0

for idx, (hx, hc, hl) in enumerate(zip(head_x_positions, head_colors, head_labels)):
    # Q, K, V boxes
    for j, label in enumerate(['Q', 'K', 'V']):
        bx = hx - 0.6 + j * 0.6
        box = mpatches.FancyBboxPatch((bx, y_proj), 0.5, 0.5, boxstyle='round,pad=0.05',
                                      facecolor=hc, edgecolor='black', alpha=0.3, linewidth=1)
        ax.add_patch(box)
        ax.text(bx + 0.25, y_proj + 0.25, label, ha='center', va='center', fontsize=8, fontweight='bold')
    # Arrow from input to projections
    ax.annotate('', xy=(hx + 0.2, y_proj), xytext=(7, 0.8),
                arrowprops=dict(arrowstyle='->', color=hc, lw=1.2, alpha=0.6))
    # Attention box
    attn_box = mpatches.FancyBboxPatch((hx - 0.8, y_attn), 2, 0.8, boxstyle='round,pad=0.1',
                                       facecolor=hc, edgecolor='black', alpha=0.4, linewidth=1.5)
    ax.add_patch(attn_box)
    ax.text(hx + 0.2, y_attn + 0.4, hl, ha='center', va='center', fontsize=10, fontweight='bold')
    # Arrow from projections to attention
    ax.annotate('', xy=(hx + 0.2, y_attn), xytext=(hx + 0.2, y_proj + 0.5),
                arrowprops=dict(arrowstyle='->', color=hc, lw=1.5))
    # Dots between head 3 and head h
    if idx == 2:
        ax.text(hx + 2.0, y_attn + 0.35, '...', fontsize=18, ha='center', va='center')

# Concat box
y_concat = 5.8
concat_box = mpatches.FancyBboxPatch((4.5, y_concat), 5, 0.8, boxstyle='round,pad=0.1',
                                      facecolor='#D6E4F0', edgecolor=COLORS['blue'], linewidth=2)
ax.add_patch(concat_box)
ax.text(7, y_concat + 0.4, 'Concat', ha='center', va='center', fontsize=12, fontweight='bold',
        color=COLORS['blue'])

for hx, hc in zip(head_x_positions, head_colors):
    ax.annotate('', xy=(7, y_concat), xytext=(hx + 0.2, y_attn + 0.8),
                arrowprops=dict(arrowstyle='->', color=hc, lw=1.5))

# Linear projection output
y_linear = 7.0
lin_box = mpatches.FancyBboxPatch((5.0, y_linear), 4, 0.7, boxstyle='round,pad=0.1',
                                   facecolor='#E8F5E9', edgecolor=COLORS['green'], linewidth=2)
ax.add_patch(lin_box)
ax.text(7, y_linear + 0.35, 'Linear (W_O)', ha='center', va='center', fontsize=12,
        fontweight='bold', color=COLORS['green'])
ax.annotate('', xy=(7, y_linear), xytext=(7, y_concat + 0.8),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

ax.set_title('Multi-Head Attention', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig12_8_multihead')
plt.show()

## Figure 12.11 -- Positional Encoding Heatmap

Sinusoidal positional encoding matrix (position x dimension).

In [ ]:
max_len, d_model = 64, 128
pe = np.zeros((max_len, d_model))
position = np.arange(max_len)[:, np.newaxis]
div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
pe[:, 0::2] = np.sin(position * div_term)
pe[:, 1::2] = np.cos(position * div_term)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pe, cmap='RdBu_r', aspect='auto', interpolation='nearest')
ax.set_xlabel('Encoding Dimension $d$', fontsize=12)
ax.set_ylabel('Position $t$', fontsize=12)
ax.set_title('Sinusoidal Positional Encoding', fontweight='bold', fontsize=14)
cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Value', fontsize=10)

fig.tight_layout()
save_fig(fig, 'fig12_11_positional_encoding')
plt.show()

## Figure 12 -- Transformer Encoder Block

Input -> Multi-Head Attention -> Add & Norm -> FFN -> Add & Norm -> Output.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 12))
ax.set_xlim(-3, 7)
ax.set_ylim(-1, 15)
ax.axis('off')

def draw_block(ax, x, y, w, h, label, facecolor, edgecolor, fontsize=11):
    box = mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                                   facecolor=facecolor, edgecolor=edgecolor, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color=edgecolor)

cx = 2  # center x
bw = 4  # box width
bx = cx - bw/2  # box left

# Input
draw_block(ax, bx, 0, bw, 0.8, 'Input Embedding\n+ Positional Encoding', '#E8E8E8', '#333333')

# Arrow up
ax.annotate('', xy=(cx, 2.0), xytext=(cx, 0.8),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Multi-Head Attention
draw_block(ax, bx, 2.0, bw, 1.0, 'Multi-Head\nSelf-Attention', '#D6E4F0', COLORS['blue'])

# Residual arrow (bypass)
ax.annotate('', xy=(cx + bw/2 + 0.8, 4.2), xytext=(cx + bw/2 + 0.8, 1.8),
            arrowprops=dict(arrowstyle='->', color=COLORS['amber'], lw=1.5, linestyle='--'))
ax.text(cx + bw/2 + 1.2, 3.0, 'residual', fontsize=8, color=COLORS['amber'], rotation=90,
        ha='center', va='center')

# Arrow
ax.annotate('', xy=(cx, 4.2), xytext=(cx, 3.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Add symbol
circle1 = plt.Circle((cx, 3.6), 0.25, facecolor='white', edgecolor='black', linewidth=1.5, zorder=5)
ax.add_patch(circle1)
ax.text(cx, 3.6, '+', ha='center', va='center', fontsize=14, fontweight='bold', zorder=6)

# Add & Norm 1
draw_block(ax, bx, 4.2, bw, 0.7, 'Add & LayerNorm', '#FFF3E0', COLORS['amber'])

# Arrow
ax.annotate('', xy=(cx, 6.1), xytext=(cx, 4.9),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# FFN
draw_block(ax, bx, 6.1, bw, 1.0, 'Feed-Forward\nNetwork', '#E8F5E9', COLORS['green'])

# Residual arrow (bypass)
ax.annotate('', xy=(cx + bw/2 + 0.8, 8.3), xytext=(cx + bw/2 + 0.8, 5.9),
            arrowprops=dict(arrowstyle='->', color=COLORS['amber'], lw=1.5, linestyle='--'))
ax.text(cx + bw/2 + 1.2, 7.1, 'residual', fontsize=8, color=COLORS['amber'], rotation=90,
        ha='center', va='center')

# Arrow
ax.annotate('', xy=(cx, 8.3), xytext=(cx, 7.1),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Add symbol
circle2 = plt.Circle((cx, 7.7), 0.25, facecolor='white', edgecolor='black', linewidth=1.5, zorder=5)
ax.add_patch(circle2)
ax.text(cx, 7.7, '+', ha='center', va='center', fontsize=14, fontweight='bold', zorder=6)

# Add & Norm 2
draw_block(ax, bx, 8.3, bw, 0.7, 'Add & LayerNorm', '#FFF3E0', COLORS['amber'])

# Arrow to output
ax.annotate('', xy=(cx, 10.2), xytext=(cx, 9.0),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# Output
draw_block(ax, bx, 10.2, bw, 0.8, 'Output', '#E8E8E8', '#333333')

ax.set_title('Transformer Encoder Block', fontweight='bold', fontsize=15, pad=15)

fig.tight_layout()
save_fig(fig, 'fig12_transformer_block')
plt.show()

## Summary

- **fig12_1**: Cross-attention word alignment between source and target sentences
- **fig12_7**: Scaled dot-product attention weight heatmap
- **fig12_8**: Multi-head attention architecture diagram
- **fig12_11**: Sinusoidal positional encoding heatmap
- **fig12_transformer_block**: Transformer encoder block with residual connections